# Lecture 07 — Plotting

*Turning numbers into clear figures: spectra, calibration curves and property plots.*

## Learning objectives

By the end of this lecture you will be able to:

- make **line**, **scatter**, **bar** and **histogram** plots with `matplotlib.pyplot`;
- label axes (**with units**), add a **title**, a **legend** and set axis **limits**;
- put **multiple series** on one set of axes and control the **figure size**;
- **save** a figure to a file with `plt.savefig`;
- explain what makes a tidy, publication-style scientific figure.

## Recap of Lecture 06

- **NumPy arrays** do fast, element-wise maths; we used them for Beer–Lambert and descriptors.
- We loaded a **UV–Vis spectrum** and found its peak with `argmax`.
- **Boolean masks** select array elements that pass a test.

## The golden rule of scientific figures

A figure is only useful if a reader can tell what it shows. So, every single time:

- **label both axes**, and **include the units** (e.g. "wavelength / nm");
- give it a **title**;
- add a **legend** if there is more than one series.

We will hold to this throughout. Let us load our data first (the spectrum and the molecule table from earlier lectures).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

spectrum = pd.read_csv("../data/uvvis_spectrum.csv")
wavelength = spectrum["wavelength_nm"].to_numpy()
absorbance = spectrum["absorbance"].to_numpy()

molecules = pd.read_csv("../data/molecules.csv")
print("Loaded", len(wavelength), "spectrum points and", len(molecules), "molecules")

## A line plot: the UV–Vis spectrum

The basic recipe: create a figure, call `plt.plot(x, y)`, then label everything and `plt.show()`.

In [ ]:
plt.figure(figsize=(7, 4))            # width, height in inches
plt.plot(wavelength, absorbance)
plt.xlabel("Wavelength / nm")
plt.ylabel("Absorbance")
plt.title("Synthetic UV-Vis spectrum")
plt.show()

### Annotating the peak

We can mark the peak using `argmax` from Lecture 06, and `plt.annotate` to add a label with an arrow.

In [ ]:
peak_i = absorbance.argmax()
peak_x = wavelength[peak_i]
peak_y = absorbance[peak_i]

plt.figure(figsize=(7, 4))
plt.plot(wavelength, absorbance, color="darkblue")
plt.annotate(f"peak at {peak_x:.0f} nm",
             xy=(peak_x, peak_y),
             xytext=(peak_x + 30, peak_y * 0.9),
             arrowprops={"arrowstyle": "->"})
plt.xlabel("Wavelength / nm")
plt.ylabel("Absorbance")
plt.title("UV-Vis spectrum with peak annotated")
plt.show()

### 🔬 Try it yourself

Plot the same spectrum but (1) change the line **colour**, (2) restrict the x-axis to **240–300 nm** with `plt.xlim(240, 300)`, and (3) give it your own title.

In [ ]:
# Your code here.

**Solution**

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(wavelength, absorbance, color="crimson")
plt.xlim(240, 300)
plt.xlabel("Wavelength / nm")
plt.ylabel("Absorbance")
plt.title("Zoom on the absorption band")
plt.show()

## A scatter plot with a best-fit line: Beer–Lambert calibration

> **🧪 Chemistry aside — calibration curve.** To measure an unknown concentration, you first build a **calibration curve**: prepare standards of known concentration, measure each one's absorbance, and fit a straight line (Beer–Lambert predicts `A = ε·l·c`, a line through the origin). You then read unknowns off that line.

We make some standards, add a touch of realistic scatter, plot them as points, and overlay a best-fit line found with `np.polyfit`.

In [ ]:
rng = np.random.default_rng(0)        # fixed seed -> reproducible "measurements"
conc = np.linspace(0.0, 1.0e-4, 8)    # mol/L (0 to 100 micromolar)
true_slope = 12000                    # epsilon * path length, L/(mol*cm)
measured_A = true_slope * conc + rng.normal(0, 0.03, size=conc.size)

# Fit a straight line: returns [slope, intercept]
slope, intercept = np.polyfit(conc, measured_A, 1)
fit_line = slope * conc + intercept

plt.figure(figsize=(6, 4))
plt.scatter(conc, measured_A, label="measured standards")
plt.plot(conc, fit_line, color="black", label=f"best fit (slope {slope:.0f})")
plt.xlabel("Concentration / mol L$^{-1}$")
plt.ylabel("Absorbance")
plt.title("Beer-Lambert calibration curve")
plt.legend()
plt.show()

## A scatter coloured by a category: molar mass vs logP

We can split points by a property. Here we group molecules by whether they are drug-like — the **rule of five from Lecture 04** (at most one violation) — tying the chemistry from the whole course together.

> **A good-figure habit.** We deliberately avoid the classic **red/green** pairing, which is the hardest combination for colour-blind readers to tell apart. Instead we use a colour-blind-friendly blue/orange pair **and** different marker *shapes* (circles vs crosses), so the two groups are distinguishable even in greyscale or for colour-blind readers.

In [ ]:
mw = molecules["molar_mass"].to_numpy()
logp = molecules["logp"].to_numpy()
donors = molecules["h_bond_donors"].to_numpy()
acceptors = molecules["h_bond_acceptors"].to_numpy()

# Same rule of five as Lecture 04: count violations, drug-like if at most one.
violations = ((mw > 500).astype(int) + (logp > 5).astype(int)
              + (donors > 5).astype(int) + (acceptors > 10).astype(int))
drug_like = violations <= 1

plt.figure(figsize=(6, 4))
plt.scatter(mw[drug_like], logp[drug_like],
            color="tab:blue", marker="o", s=60, label="drug-like")
plt.scatter(mw[~drug_like], logp[~drug_like],
            color="tab:orange", marker="X", s=90, label="not drug-like")
plt.xlabel("Molar mass / g mol$^{-1}$")
plt.ylabel("logP (computed)")
plt.title("Molecular properties by drug-likeness (Lipinski)")
plt.legend()
plt.show()

## A histogram and a bar chart

A **histogram** shows the distribution of one set of values (how many fall in each range). A **bar chart** compares a value across labelled categories.

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(mw, bins=6, color="slateblue", edgecolor="white")
plt.xlabel("Molar mass / g mol$^{-1}$")
plt.ylabel("Number of molecules")
plt.title("Distribution of molar masses")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(molecules["name"], molecules["molar_mass"], color="teal")
plt.xlabel("Molecule")
plt.ylabel("Molar mass / g mol$^{-1}$")
plt.title("Molar mass of each molecule")
plt.xticks(rotation=45, ha="right")     # rotate labels so they do not overlap
plt.tight_layout()
plt.show()

### 🔬 Try it yourself

Make a **bar chart of logP** for each molecule (use `molecules["logp"]`). Label the axes (logP has no units), add a title, and rotate the x-tick labels as above so the names stay readable.

In [ ]:
# Your code here.

**Solution**

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(molecules["name"], molecules["logp"], color="darkorange")
plt.xlabel("Molecule")
plt.ylabel("logP (computed)")
plt.title("Computed logP of each molecule")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 🔬 Try it yourself

The plot below runs, but it is **missing its labels** — a cardinal sin for a scientific figure. In the solution cell, take the same plot and add an **x-label** (`"Wavelength / nm"`), a **y-label** (`"Absorbance"`), a **title**, and a **legend**.

In [ ]:
# A deliberately unlabelled plot — fix it in the solution below.
plt.figure(figsize=(7, 4))
plt.plot(wavelength, absorbance)
plt.show()

**Solution**

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(wavelength, absorbance, label="UV-Vis spectrum")
plt.xlabel("Wavelength / nm")
plt.ylabel("Absorbance")
plt.title("UV-Vis absorption spectrum")
plt.legend()
plt.show()

## Saving a figure

To keep a figure, call `plt.savefig(...)` **before** `plt.show()`. The file extension chooses the format (`.png`, `.pdf`, ...). `dpi` controls the resolution.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(wavelength, absorbance, color="darkblue")
plt.xlabel("Wavelength / nm")
plt.ylabel("Absorbance")
plt.title("UV-Vis spectrum")
plt.savefig("uvvis_spectrum.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved uvvis_spectrum.png")

### 🔬 Try it yourself

Make **one** figure with **two series on the same axes**: the raw spectrum and its **normalised** version (`absorbance / absorbance.max()`). Add a **legend** so the two lines are distinguishable, label the axes, and save it as a PNG.

In [ ]:
# Your code here.

**Solution**

In [ ]:
normalised = absorbance / absorbance.max()

plt.figure(figsize=(7, 4))
plt.plot(wavelength, absorbance, label="raw")
plt.plot(wavelength, normalised, label="normalised")
plt.xlabel("Wavelength / nm")
plt.ylabel("Absorbance (raw or normalised)")
plt.title("Raw vs normalised spectrum")
plt.legend()
plt.savefig("spectrum_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## ⚗️ With RDKit — show the molecules behind the numbers

Finally, let us put faces to the data. `Draw.MolsToGridImage` renders our molecule set as a labelled grid — the structures whose properties we have been plotting all along.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

names = molecules["name"].tolist()
mols = [Chem.MolFromSmiles(smi) for smi in molecules["smiles"]]

Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(180, 160), legends=names)

## Key takeaways

- `matplotlib.pyplot` makes **line**, **scatter**, **bar** and **histogram** plots.
- **Always** label both axes with units, add a title, and use a legend for multiple series.
- Put several series on one axes by calling `plot`/`scatter` more than once before `show`.
- `plt.figure(figsize=...)` sets the size; `plt.savefig(...)` (before `show`) saves the figure.
- RDKit's `Draw.MolsToGridImage` displays the molecules behind the numbers.

## Where to go next

Congratulations — you have covered the Python fundamentals a working chemist needs, all the way from a bare notebook to publication-style figures, with RDKit woven throughout. To keep going:

- **Real datasets.** Try your skills on actual data — for example public sets from PubChem or ChEMBL — instead of our small teaching table.
- **Deeper RDKit.** There is a great deal more: molecular **fingerprints**, **substructure search** (does my molecule contain this group?), and **reaction** handling.
- **The browser version.** This workshop also exists as **marimo** notebooks that run entirely in the browser via WebAssembly — handy for experimenting with the Python without installing anything (note that RDKit is not available in that in-browser version, which is exactly why we used local Jupyter here).

Most of all: keep using it. The fastest way to get comfortable is to automate a small, real task from your own work.